In [0]:
%run ./sentinelmesh_agent.py

In [0]:
print(run_triage("Network flow alert: ICMP dominant traffic, 16529 packets/sec, uniform packet sizes, no TCP flags consistently set."))

**Triage Summary**

- **Predicted Attack Family:** DDOS_FLOOD  
- **MITRE ATT&CK Technique:** T1498 – Network Denial of Service  
- **Severity Score:** 10 / 10 (critical)  
- **Recommended Action:** Immediate containment. Apply rate limiting, upstream traffic filtering, and engage DDoS protection services as per MITRE guidance. Await analyst approval before execution.


[Trace(trace_id=tr-8822eda1b176982c2d1d807202a9b768), Trace(trace_id=tr-0098aaa7e23cf190c4423329735debd9), Trace(trace_id=tr-dd6b4ff8d2a4035a188a4c9805668238), Trace(trace_id=tr-317422a601c1b0b52a56282b1d48f6c6)]

In [0]:
import warnings
warnings.filterwarnings("ignore")

sentinelmesh_agent.py loaded successfully.


In [0]:
## Build the MITRE ATT&CK Reference Table

# This is our threat-intelligence lookup. Each of the 34 CIC-IoT-2023 attack
# types is mapped to its official MITRE ATT&CK technique, plus a short
# description, a detection idea, and a mitigation. The agent's enrichment tool
# will query this table (by attack family) instead of guessing from memory --
# that's what makes it a real tool grounded in structured threat intel.

import pandas as pd

# each row: the CIC-IoT label family -> MITRE technique + context
# (grouped by attack family so we cover all 34 labels without 34 separate rows)
mitre_data = [
    # --- DDoS / DoS flood family (volumetric network attacks) ---
    {"attack_family": "DDOS_FLOOD", "technique_id": "T1498",
     "technique_name": "Network Denial of Service",
     "description": "Attacker overwhelms a target with high-volume traffic to exhaust bandwidth or resources.",
     "detection": "Look for sustained abnormally high packet rates and uniform packet sizes from one or many sources.",
     "mitigation": "Rate limiting, upstream traffic filtering, and DDoS protection services."},

    {"attack_family": "DOS_FLOOD", "technique_id": "T1499",
     "technique_name": "Endpoint Denial of Service",
     "description": "Attacker exhausts the resources of a single endpoint or service to make it unavailable.",
     "detection": "High request rate targeting one host; resource exhaustion on the endpoint.",
     "mitigation": "Connection limits, resource quotas, and application-layer filtering."},

    # --- Mirai botnet family ---
    {"attack_family": "MIRAI", "technique_id": "T1498",
     "technique_name": "Network Denial of Service (Botnet)",
     "description": "Compromised IoT devices form a botnet that floods targets with traffic.",
     "detection": "Coordinated high-rate traffic from many IoT devices; known Mirai traffic signatures.",
     "mitigation": "Device hardening, default-credential changes, and network segmentation."},

    # --- Reconnaissance family ---
    {"attack_family": "RECON", "technique_id": "T1595",
     "technique_name": "Active Scanning",
     "description": "Attacker probes the network to discover hosts, open ports, and services before attacking.",
     "detection": "Low-rate connections spread across many ports or hosts; scanning patterns.",
     "mitigation": "Firewall rules, port-scan detection, and limiting externally exposed services."},

    {"attack_family": "VULNERABILITYSCAN", "technique_id": "T1595.002",
     "technique_name": "Vulnerability Scanning",
     "description": "Attacker scans systems for known vulnerabilities to exploit.",
     "detection": "Repeated probes matching known vulnerability-scanner signatures.",
     "mitigation": "Patch management and intrusion prevention systems."},

    # --- Spoofing / Man-in-the-middle family ---
    {"attack_family": "SPOOFING", "technique_id": "T1557",
     "technique_name": "Adversary-in-the-Middle",
     "description": "Attacker intercepts or redirects traffic by impersonating a legitimate host (ARP/DNS spoofing).",
     "detection": "Duplicate or conflicting ARP/DNS responses; unexpected gateway changes.",
     "mitigation": "Dynamic ARP inspection, DNS security (DNSSEC), and network monitoring."},

    # --- Brute force family ---
    {"attack_family": "BRUTEFORCE", "technique_id": "T1110",
     "technique_name": "Brute Force",
     "description": "Attacker repeatedly guesses credentials to gain unauthorized access.",
     "detection": "Many failed authentication attempts in a short window from one source.",
     "mitigation": "Account lockouts, rate limiting, and multi-factor authentication."},

    # --- Web application attack family ---
    {"attack_family": "WEB_ATTACK", "technique_id": "T1190",
     "technique_name": "Exploitation of Public-Facing Application",
     "description": "Attacker exploits a web application flaw (SQL injection, XSS, command injection, uploads).",
     "detection": "Malformed requests, injection patterns, or unexpected payloads in web traffic.",
     "mitigation": "Input validation, web application firewalls, and secure coding practices."},

    # --- Malware / backdoor family ---
    {"attack_family": "MALWARE", "technique_id": "T1059",
     "technique_name": "Command and Scripting Interpreter",
     "description": "Traffic consistent with malware or backdoor command-and-control activity. Note: this is an approximate behavioral mapping, flow statistics suggest but do not prove command execution.",
     "detection": "Unusual outbound connections; low-and-slow traffic that mimics benign flows.",
     "mitigation": "Endpoint detection, egress filtering, and behavioral analysis."},

    # --- Benign (not an attack) ---
    {"attack_family": "BENIGN", "technique_id": "N/A",
     "technique_name": "Benign Traffic",
     "description": "Normal, expected IoT device behavior with no malicious indicators.",
     "detection": "Traffic matches established baselines for the device type.",
     "mitigation": "No action required; continue monitoring."},
]

mitre_df = pd.DataFrame(mitre_data)
print(f"MITRE reference table built: {len(mitre_df)} attack families mapped")
display(mitre_df)

MITRE reference table built: 10 attack families mapped


attack_family,technique_id,technique_name,description,detection,mitigation
DDOS_FLOOD,T1498,Network Denial of Service,Attacker overwhelms a target with high-volume traffic to exhaust bandwidth or resources.,Look for sustained abnormally high packet rates and uniform packet sizes from one or many sources.,"Rate limiting, upstream traffic filtering, and DDoS protection services."
DOS_FLOOD,T1499,Endpoint Denial of Service,Attacker exhausts the resources of a single endpoint or service to make it unavailable.,High request rate targeting one host; resource exhaustion on the endpoint.,"Connection limits, resource quotas, and application-layer filtering."
MIRAI,T1498,Network Denial of Service (Botnet),Compromised IoT devices form a botnet that floods targets with traffic.,Coordinated high-rate traffic from many IoT devices; known Mirai traffic signatures.,"Device hardening, default-credential changes, and network segmentation."
RECON,T1595,Active Scanning,"Attacker probes the network to discover hosts, open ports, and services before attacking.",Low-rate connections spread across many ports or hosts; scanning patterns.,"Firewall rules, port-scan detection, and limiting externally exposed services."
VULNERABILITYSCAN,T1595.002,Vulnerability Scanning,Attacker scans systems for known vulnerabilities to exploit.,Repeated probes matching known vulnerability-scanner signatures.,Patch management and intrusion prevention systems.
SPOOFING,T1557,Adversary-in-the-Middle,Attacker intercepts or redirects traffic by impersonating a legitimate host (ARP/DNS spoofing).,Duplicate or conflicting ARP/DNS responses; unexpected gateway changes.,"Dynamic ARP inspection, DNS security (DNSSEC), and network monitoring."
BRUTEFORCE,T1110,Brute Force,Attacker repeatedly guesses credentials to gain unauthorized access.,Many failed authentication attempts in a short window from one source.,"Account lockouts, rate limiting, and multi-factor authentication."
WEB_ATTACK,T1190,Exploitation of Public-Facing Application,"Attacker exploits a web application flaw (SQL injection, XSS, command injection, uploads).","Malformed requests, injection patterns, or unexpected payloads in web traffic.","Input validation, web application firewalls, and secure coding practices."
MALWARE,T1059,Command and Scripting Interpreter,"Traffic consistent with malware or backdoor command-and-control activity. Note: this is an approximate behavioral mapping, flow statistics suggest but do not prove command execution.",Unusual outbound connections; low-and-slow traffic that mimics benign flows.,"Endpoint detection, egress filtering, and behavioral analysis."
BENIGN,N/A,Benign Traffic,"Normal, expected IoT device behavior with no malicious indicators.",Traffic matches established baselines for the device type.,No action required; continue monitoring.


In [0]:
## Save the MITRE Reference Table to Unity Catalog

# Persist it as a governed table so the agent's tool can query it by SQL,
# the same way your Assignment 3 tools queried Unity Catalog data.
spark.createDataFrame(mitre_df).write.format("delta").mode("overwrite") \
    .saveAsTable("main.default.mitre_reference")

print("Saved to main.default.mitre_reference")
display(spark.table("main.default.mitre_reference"))

Saved to main.default.mitre_reference


attack_family,technique_id,technique_name,description,detection,mitigation
DDOS_FLOOD,T1498,Network Denial of Service,Attacker overwhelms a target with high-volume traffic to exhaust bandwidth or resources.,Look for sustained abnormally high packet rates and uniform packet sizes from one or many sources.,"Rate limiting, upstream traffic filtering, and DDoS protection services."
DOS_FLOOD,T1499,Endpoint Denial of Service,Attacker exhausts the resources of a single endpoint or service to make it unavailable.,High request rate targeting one host; resource exhaustion on the endpoint.,"Connection limits, resource quotas, and application-layer filtering."
MIRAI,T1498,Network Denial of Service (Botnet),Compromised IoT devices form a botnet that floods targets with traffic.,Coordinated high-rate traffic from many IoT devices; known Mirai traffic signatures.,"Device hardening, default-credential changes, and network segmentation."
RECON,T1595,Active Scanning,"Attacker probes the network to discover hosts, open ports, and services before attacking.",Low-rate connections spread across many ports or hosts; scanning patterns.,"Firewall rules, port-scan detection, and limiting externally exposed services."
VULNERABILITYSCAN,T1595.002,Vulnerability Scanning,Attacker scans systems for known vulnerabilities to exploit.,Repeated probes matching known vulnerability-scanner signatures.,Patch management and intrusion prevention systems.
SPOOFING,T1557,Adversary-in-the-Middle,Attacker intercepts or redirects traffic by impersonating a legitimate host (ARP/DNS spoofing).,Duplicate or conflicting ARP/DNS responses; unexpected gateway changes.,"Dynamic ARP inspection, DNS security (DNSSEC), and network monitoring."
BRUTEFORCE,T1110,Brute Force,Attacker repeatedly guesses credentials to gain unauthorized access.,Many failed authentication attempts in a short window from one source.,"Account lockouts, rate limiting, and multi-factor authentication."
WEB_ATTACK,T1190,Exploitation of Public-Facing Application,"Attacker exploits a web application flaw (SQL injection, XSS, command injection, uploads).","Malformed requests, injection patterns, or unexpected payloads in web traffic.","Input validation, web application firewalls, and secure coding practices."
MALWARE,T1059,Command and Scripting Interpreter,"Traffic consistent with malware or backdoor command-and-control activity. Note: this is an approximate behavioral mapping, flow statistics suggest but do not prove command execution.",Unusual outbound connections; low-and-slow traffic that mimics benign flows.,"Endpoint detection, egress filtering, and behavioral analysis."
BENIGN,N/A,Benign Traffic,"Normal, expected IoT device behavior with no malicious indicators.",Traffic matches established baselines for the device type.,No action required; continue monitoring.


In [0]:
## Build the CIC Label -> Attack Family Map (Unity Catalog Table)

# Maps each of the 34 granular CICIoT2023 labels to one of our MITRE families.
# Saved as a governed table (not a Python dict) so it's queryable and
# course-aligned, the same pattern as your Assignment 3 Unity Catalog tools.
import pandas as pd

label_family_rows = [
    # DDoS floods
    ("DDOS-ICMP_FLOOD","DDOS_FLOOD"), ("DDOS-UDP_FLOOD","DDOS_FLOOD"),
    ("DDOS-TCP_FLOOD","DDOS_FLOOD"), ("DDOS-PSHACK_FLOOD","DDOS_FLOOD"),
    ("DDOS-RSTFINFLOOD","DDOS_FLOOD"), ("DDOS-SYN_FLOOD","DDOS_FLOOD"),
    ("DDOS-SYNONYMOUSIP_FLOOD","DDOS_FLOOD"), ("DDOS-ICMP_FRAGMENTATION","DDOS_FLOOD"),
    ("DDOS-UDP_FRAGMENTATION","DDOS_FLOOD"), ("DDOS-ACK_FRAGMENTATION","DDOS_FLOOD"),
    ("DDOS-HTTP_FLOOD","DDOS_FLOOD"), ("DDOS-SLOWLORIS","DDOS_FLOOD"),
    # DoS floods
    ("DOS-UDP_FLOOD","DOS_FLOOD"), ("DOS-TCP_FLOOD","DOS_FLOOD"),
    ("DOS-SYN_FLOOD","DOS_FLOOD"), ("DOS-HTTP_FLOOD","DOS_FLOOD"),
    # Mirai
    ("MIRAI-GREETH_FLOOD","MIRAI"), ("MIRAI-UDPPLAIN","MIRAI"), ("MIRAI-GREIP_FLOOD","MIRAI"),
    # Recon (+ vuln scan separate)
    ("RECON-HOSTDISCOVERY","RECON"), ("RECON-OSSCAN","RECON"),
    ("RECON-PORTSCAN","RECON"), ("RECON-PINGSWEEP","RECON"),
    ("VULNERABILITYSCAN","VULNERABILITYSCAN"),
    # Spoofing / MITM
    ("MITM-ARPSPOOFING","SPOOFING"), ("DNS_SPOOFING","SPOOFING"),
    # Brute force
    ("DICTIONARYBRUTEFORCE","BRUTEFORCE"),
    # Web attacks
    ("SQLINJECTION","WEB_ATTACK"), ("XSS","WEB_ATTACK"),
    ("COMMANDINJECTION","WEB_ATTACK"), ("BROWSERHIJACKING","WEB_ATTACK"),
    ("UPLOADING_ATTACK","WEB_ATTACK"),
    # Malware / backdoor
    ("BACKDOOR_MALWARE","MALWARE"),
    # Benign
    ("BENIGN","BENIGN"),
]

label_family_df = pd.DataFrame(label_family_rows, columns=["label", "attack_family"])

# save to Unity Catalog
spark.createDataFrame(label_family_df).write.format("delta").mode("overwrite") \
    .saveAsTable("main.default.cic_label_family_map")

# verify every sample label is covered (unmapped should be empty)
sample_labels = set(spark.table("main.default.cic_alerts_sample").select("Label").distinct().toPandas()["Label"])
mapped_labels = set(label_family_df["label"])
print(f"Saved main.default.cic_label_family_map: {len(label_family_df)} labels")
print(f"Unmapped labels (should be empty): {sample_labels - mapped_labels}")
display(spark.table("main.default.cic_label_family_map"))

Saved main.default.cic_label_family_map: 34 labels
Unmapped labels (should be empty): set()


label,attack_family
DDOS-ICMP_FLOOD,DDOS_FLOOD
DDOS-UDP_FLOOD,DDOS_FLOOD
DDOS-TCP_FLOOD,DDOS_FLOOD
DDOS-PSHACK_FLOOD,DDOS_FLOOD
DDOS-RSTFINFLOOD,DDOS_FLOOD
DDOS-SYN_FLOOD,DDOS_FLOOD
DDOS-SYNONYMOUSIP_FLOOD,DDOS_FLOOD
DDOS-ICMP_FRAGMENTATION,DDOS_FLOOD
DDOS-UDP_FRAGMENTATION,DDOS_FLOOD
DDOS-ACK_FRAGMENTATION,DDOS_FLOOD


In [0]:
## First Test: Triage One Real Alert

# pull one alert from your sample table -- only the alert_text goes to the agent,
# never the Label. we keep the true label aside just to see how it did.
sample = spark.table("main.default.cic_alerts_sample").toPandas()
test_row = sample.iloc[1]   # try different indexes to test different attacks

print("ALERT GIVEN TO AGENT:")
print(test_row["alert_text"])
print("\n" + "="*70)
print("AGENT TRIAGE DECISION:")
print(run_triage(test_row["alert_text"]))
print("="*70)
print(f"\n(TRUE LABEL, hidden from agent: {test_row['Label']})")

ALERT GIVEN TO AGENT:
Network flow alert: IPv dominant traffic (100% of flow), packets per second: 34, packet count: 10, average packet length: 140 bytes, packet length standard deviation: 70, minimum packet length: 60, maximum packet length: 218, total packet size: 140, total packet size sum: 1400, inter arrival time: 0.030426 seconds, SYN count: 0, ACK count: 4, RST count: 0, FIN count: 0, header length: 18, time to live: 99, variance: 4831.11, varied packet sizes, TCP flag activity: ACK=0.40, PSH=0.20. The true label is not shown to the agent during evaluation.

AGENT TRIAGE DECISION:
**Triage Summary**

- **Predicted Attack Family:** RECON  
- **MITRE ATT&CK Technique:** T1595 – Active Scanning (attacker probing the network to discover hosts, open ports, and services)  
- **Severity Score:** 4 / 10 (Low) – modest packet rate (34 pps) and mixed ACK/PSH flags suggest a low‑intensity reconnaissance activity.  
- **Recommended Action:** **Low urgency.** Monitor and log the traffic for 

[Trace(trace_id=tr-b431478f228e0ff11d611078f804d108), Trace(trace_id=tr-0a66b0efe872839b4691f8c748f2b7b5), Trace(trace_id=tr-3d308ec58869be0eeb02537d98623ab9), Trace(trace_id=tr-f1cb90efdafe18a76b196090904d01ce)]

In [0]:
## Enable MLflow Tracing

# Autolog captures every LLM call and tool call as a structured trace.
# Each triage run becomes a viewable trace in the MLflow Experiments UI,
# giving full visibility into the agent's reasoning and tool usage.
import mlflow

mlflow.openai.autolog()   # auto traces the OpenAI style client calls
mlflow.set_experiment("/Users/" + spark.sql("SELECT current_user()").first()[0] + "/sentinelmesh_traces")

print("MLflow tracing enabled.")

MLflow tracing enabled.


In [0]:
## Wrap Triage in a Traced Function

# A thin wrapper that records each triage as a single named trace, tagged
# with the true label so prediction can later be compared against ground
# truth. The agent itself never sees the label; it is stored on the trace
# only for evaluation purposes.
@mlflow.trace(name="sentinelmesh_triage")
def traced_triage(alert_text, true_label, model=LLM_ENDPOINT):
    # record the true label and model on the trace metadata for later review
    mlflow.update_current_trace(tags={"true_label": true_label, "model": model})

    # run the actual agent
    decision = run_triage(alert_text, model=model)
    return decision

print("Traced triage wrapper ready.")

Traced triage wrapper ready.


In [0]:
## Confirm Tracing Works

# Run a single alert through the traced wrapper and confirm a trace is
# created in the Experiments UI.
sample = spark.table("main.default.cic_alerts_sample").toPandas()
row = sample.iloc[20]

decision = traced_triage(row["alert_text"], row["Label"])
print("Decision:", decision[:300], "...")
print(f"\nTrue label: {row['Label']}")
print("\nA new 'sentinelmesh_triage' trace should now appear in the Experiments tab.")

Decision: **Triage Summary**

- **Predicted Attack Family:** **DOS_FLOOD**  
- **MITRE ATT&CK Technique:** **T1499 – Endpoint Denial of Service**  
  - *Description:* Exhausts resources of a single endpoint or service, rendering it unavailable.  
  - *Detection:* Unusually high request rate targeting one host ...

True label: DDOS-ACK_FRAGMENTATION

A new 'sentinelmesh_triage' trace should now appear in the Experiments tab.


Trace(trace_id=tr-6ba261719298a171eaa2f043dcb8a5dc)

In [0]:
## Select Representative Alerts for Tracing

# Pull a few alerts from different attack families so the traces show the
# agent handling distinct attack types, not the same one repeatedly.
sample = spark.table("main.default.cic_alerts_sample").toPandas()

# show what families are available with their row indexes, to choose from
for family in ["DDOS-ICMP_FLOOD", "RECON-PORTSCAN", "MITM-ARPSPOOFING", "BENIGN", "DICTIONARYBRUTEFORCE"]:
    matches = sample[sample["Label"] == family]
    if len(matches) > 0:
        idx = matches.index[0]
        print(f"Row {idx}: {family}")
        print(f"   {sample.loc[idx, 'alert_text']}\n")

Row 29: DDOS-ICMP_FLOOD
   Network flow alert: ICMP dominant traffic (100% of flow), packets per second: 16,529, packet count: 100, average packet length: 60 bytes, packet length standard deviation: 0, minimum packet length: 60, maximum packet length: 60, total packet size: 60, total packet size sum: 6000, inter arrival time: 0.000061 seconds, SYN count: 0, ACK count: 0, RST count: 0, FIN count: 0, header length: 0, time to live: 64, variance: 0.00, highly uniform packet sizes, TCP flag activity: no TCP flags observed. The true label is not shown to the agent during evaluation.

Row 144: RECON-PORTSCAN
   Network flow alert: IPv dominant traffic (100% of flow), packets per second: 121, packet count: 10, average packet length: 1394 bytes, packet length standard deviation: 1241, minimum packet length: 66, maximum packet length: 2962, total packet size: 1394, total packet size sum: 13938, inter arrival time: 0.015494 seconds, SYN count: 1, ACK count: 8, RST count: 0, FIN count: 0, header 

In [0]:
## Trace 1 and Trace 2: Normal Triage on Two Different Attacks

# Two standard triage runs on distinct attack families. Each is recorded as a
# trace showing the full reasoning and tool sequence.
sample = spark.table("main.default.cic_alerts_sample").toPandas()

# Trace 1: a high volume flood attack
flood_row = sample[sample["Label"] == "DDOS-ICMP_FLOOD"].iloc[0]
print("TRACE 1 -", flood_row["Label"])
print(traced_triage(flood_row["alert_text"], flood_row["Label"]))
print("\n" + "="*70 + "\n")

# Trace 2: a reconnaissance attack (different family, different signature)
recon_row = sample[sample["Label"] == "RECON-PORTSCAN"].iloc[0]
print("TRACE 2 -", recon_row["Label"])
print(traced_triage(recon_row["alert_text"], recon_row["Label"]))

TRACE 1 - DDOS-ICMP_FLOOD
**Triage Summary**

- **Predicted Attack Family:** DDOS_FLOOD  
- **MITRE ATT&CK Technique:** T1498 – Network Denial of Service (ICMP flood)  
- **Severity Score:** 10 / 10 (Critical) – Very high packet rate (16,529 pps) with uniform 60‑byte ICMP packets indicates a massive flood.  
- **Recommended Action:** *CRITICAL* – Implement immediate containment: apply rate‑limiting, engage upstream traffic filtering, and activate DDoS protection services as advised by MITRE. Await analyst approval before execution.


TRACE 2 - RECON-PORTSCAN
**Triage Summary**

- **Predicted Attack Family:** **BENIGN**  
- **MITRE ATT&CK Context:** Benign Traffic – normal, expected behavior with no malicious indicators.  
- **Severity Score:** **1 / 10** (informational)  
- **Recommended Action:** **[INFORMATIONAL]** No immediate response needed; continue monitoring the flow. Follow the MITRE guidance: “No action required; continue monitoring.”  

*Human analyst approval is requested b

[Trace(trace_id=tr-e928ed6cfff6773acc1aa2932f4e3d93), Trace(trace_id=tr-db491aedfca24becabfd4cd3a6486819)]

In [0]:
## Trace 3: Two Model Comparison on the Same Alert

# Run one identical alert through both the larger and smaller model so their
# triage quality, token usage, and latency can be compared directly. This
# comparison is the basis for the cost versus quality analysis later.
sample = spark.table("main.default.cic_alerts_sample").toPandas()
compare_row = sample[sample["Label"] == "DDOS-ICMP_FLOOD"].iloc[0]

print("ALERT:", compare_row["alert_text"])
print("TRUE LABEL:", compare_row["Label"])
print("\n" + "="*70)

# larger model
print("\nMODEL A: databricks-gpt-oss-120b")
print(traced_triage(compare_row["alert_text"], compare_row["Label"], model="databricks-gpt-oss-120b"))

print("\n" + "="*70)

# smaller, cheaper model
print("\nMODEL B: databricks-gpt-oss-20b")
print(traced_triage(compare_row["alert_text"], compare_row["Label"], model="databricks-gpt-oss-20b"))

ALERT: Network flow alert: ICMP dominant traffic (100% of flow), packets per second: 16,529, packet count: 100, average packet length: 60 bytes, packet length standard deviation: 0, minimum packet length: 60, maximum packet length: 60, total packet size: 60, total packet size sum: 6000, inter arrival time: 0.000061 seconds, SYN count: 0, ACK count: 0, RST count: 0, FIN count: 0, header length: 0, time to live: 64, variance: 0.00, highly uniform packet sizes, TCP flag activity: no TCP flags observed. The true label is not shown to the agent during evaluation.
TRUE LABEL: DDOS-ICMP_FLOOD


MODEL A: databricks-gpt-oss-120b
**Triage Summary**

- **Predicted Attack Family:** DDOS_FLOOD  
- **MITRE ATT&CK Technique:** T1498 – Network Denial of Service (high‑volume ICMP flood)  
- **Severity Score:** 10 / 10 (critical)  
- **Recommended Action:** *CRITICAL* – Immediate containment is advised. Implement rate limiting, upstream traffic filtering, and engage DDoS protection services as suggested

[Trace(trace_id=tr-bb0933e5952113f4fe06bd2a4960bd0e), Trace(trace_id=tr-8f7d81f9cd9a660af062216f8a144448)]

## Model Comparison Finding

The same alert was tested with the larger `databricks-gpt-oss-120b` model and the smaller `databricks-gpt-oss-20b` model. The larger model completed the tool calling workflow and produced a usable triage decision. 


In [0]:
## Trace 4 and Trace 5: Graceful Rejection of Non Alert Inputs

# Feed the agent inputs that are not security alerts and confirm it declines
# instead of forcing a triage. This demonstrates the agent recognizes the
# boundaries of its role and does not misuse its tools on unrelated requests.

# Rejection 1: a general knowledge question
print("TRACE 4 - Off topic question")
print(traced_triage("What is the best recipe for chocolate chip cookies?", true_label="NOT_AN_ALERT"))
print("\n" + "="*70 + "\n")

# Rejection 2: a coding request unrelated to security triage
print("TRACE 5 - Unrelated coding request")
print(traced_triage("Write me a Python function that sorts a list of numbers.", true_label="NOT_AN_ALERT"))

TRACE 4 - Off topic question
User is asking about a recipe, not a network security alert. Must decline politely, explain only triage security alerts.
I’m sorry, but I can only help with triaging network security alerts. If you have a network flow alert you’d like analyzed, feel free to share it!


TRACE 5 - Unrelated coding request
User asks for a Python function unrelated to network alert triage. Must politely decline.
I’m sorry, but I can only help with triaging network security alerts. If you have a network‑flow alert you’d like analyzed, please share the details and I’ll provide a triage assessment.


[Trace(trace_id=tr-978e69e9f12bfa7a3f176dd61733bc91), Trace(trace_id=tr-8b49138a21336646e5d881dad25a8ed8)]

In [0]:
## Load the Label to Family Mapping

# Pull the label-to-family lookup from Unity Catalog so predictions and true
# labels can be compared at the family level.
label_map = spark.table("main.default.cic_label_family_map").toPandas()
label_to_family = dict(zip(label_map["label"], label_map["attack_family"]))

print(f"Loaded family mapping for {len(label_to_family)} labels.")

Loaded family mapping for 34 labels.


In [0]:
## Run the Evaluation Deterministically

import re
import time
import pandas as pd

label_map = spark.table("main.default.cic_label_family_map").toPandas()
label_to_family = dict(zip(label_map["label"], label_map["attack_family"]))

sample = spark.table("main.default.cic_alerts_sample").toPandas()
sample["expected_family"] = sample["Label"].map(label_to_family)

families_to_eval = [
    "DDOS-ICMP_FLOOD",
    "DDOS-SYN_FLOOD",
    "DOS-UDP_FLOOD",
    "MIRAI-GREETH_FLOOD",
    "RECON-PORTSCAN",
    "VULNERABILITYSCAN",
    "MITM-ARPSPOOFING",
    "DNS_SPOOFING",
    "DICTIONARYBRUTEFORCE",
    "SQLINJECTION",
    "BACKDOOR_MALWARE",
    "BENIGN",
]

eval_rows = []

for family in families_to_eval:
    matches = sample[sample["Label"] == family]
    if len(matches) > 0:
        row = matches.iloc[0]
        eval_rows.append({
            "label": row["Label"],
            "inputs": {"alert_text": row["alert_text"]},
            "expectations": {"expected_family": row["expected_family"]},
        })

print(f"Evaluation dataset built with {len(eval_rows)} alerts.")

VALID_FAMILIES = [
    "DDOS_FLOOD", "DOS_FLOOD", "VULNERABILITYSCAN", "BRUTEFORCE",
    "WEB_ATTACK", "SPOOFING", "MALWARE", "MIRAI", "RECON", "BENIGN",
]

COARSE = {
    "DDOS_FLOOD": "FLOOD",
    "DOS_FLOOD": "FLOOD",
    "MIRAI": "FLOOD",
    "RECON": "RECON_SCAN",
    "VULNERABILITYSCAN": "RECON_SCAN",
    "SPOOFING": "SPOOFING",
    "BRUTEFORCE": "INTRUSION",
    "WEB_ATTACK": "INTRUSION",
    "MALWARE": "INTRUSION",
    "BENIGN": "BENIGN",
}

def extract_predicted_family(text):
    text = str(text).upper()
    for family in VALID_FAMILIES:
        pattern = rf"(?<![A-Z_]){family}(?![A-Z_])"
        if re.search(pattern, text):
            return family
    return "UNKNOWN"

eval_results = []

for i, r in enumerate(eval_rows):
    alert = r["inputs"]["alert_text"]
    expected = r["expectations"]["expected_family"]

    decision = str(run_triage(alert))
    predicted = extract_predicted_family(decision)

    exact_correct = predicted == expected
    coarse_correct = (
        predicted != "UNKNOWN" and COARSE.get(predicted) == COARSE.get(expected)
    )

    eval_results.append({
        "source_label": r["label"],
        "expected_family": expected,
        "predicted_family": predicted,
        "expected_coarse": COARSE.get(expected),
        "predicted_coarse": COARSE.get(predicted, "UNKNOWN"),
        "exact_correct": exact_correct,
        "coarse_correct": coarse_correct,
        "decision_excerpt": decision[:180],
    })

    print(
        f"[{i+1}/{len(eval_rows)}] expected={expected:18s} "
        f"predicted={predicted:18s} exact={exact_correct} coarse={coarse_correct}"
    )

    time.sleep(5)

eval_results_df = pd.DataFrame(eval_results)

display(eval_results_df)

n = len(eval_results_df)
exact_count = int(eval_results_df["exact_correct"].sum())
coarse_count = int(eval_results_df["coarse_correct"].sum())

print(f"Exact family accuracy: {exact_count}/{n} = {exact_count / n:.0%}")
print(f"Coarse triage accuracy: {coarse_count}/{n} = {coarse_count / n:.0%}")

Evaluation dataset built with 12 alerts.
[1/12] expected=DDOS_FLOOD         predicted=DDOS_FLOOD         exact=True coarse=True
[2/12] expected=DDOS_FLOOD         predicted=DDOS_FLOOD         exact=True coarse=True
[3/12] expected=DOS_FLOOD          predicted=DDOS_FLOOD         exact=False coarse=True
[4/12] expected=MIRAI              predicted=DOS_FLOOD          exact=False coarse=True
[5/12] expected=RECON              predicted=BENIGN             exact=False coarse=False
[6/12] expected=VULNERABILITYSCAN  predicted=VULNERABILITYSCAN  exact=True coarse=True
[7/12] expected=SPOOFING           predicted=DDOS_FLOOD         exact=False coarse=False
[8/12] expected=SPOOFING           predicted=WEB_ATTACK         exact=False coarse=False
[9/12] expected=BRUTEFORCE         predicted=RECON              exact=False coarse=False
[10/12] expected=WEB_ATTACK         predicted=BENIGN             exact=False coarse=False
[11/12] expected=MALWARE            predicted=BENIGN             exact=False

source_label,expected_family,predicted_family,expected_coarse,predicted_coarse,exact_correct,coarse_correct,decision_excerpt
DDOS-ICMP_FLOOD,DDOS_FLOOD,DDOS_FLOOD,FLOOD,FLOOD,true,true,"**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service (high‑volume, uniform ICMP traffic) - **Severity S"
DDOS-SYN_FLOOD,DDOS_FLOOD,DDOS_FLOOD,FLOOD,FLOOD,true,true,"**Triage Summary** - **Predicted Attack Family:** **DDOS_FLOOD** – high‑volume, uniform‑size TCP SYN packets (44,502 pps) indicate a Distributed Denial‑of‑Service SYN flood. - *"
DOS-UDP_FLOOD,DOS_FLOOD,DDOS_FLOOD,FLOOD,FLOOD,false,true,**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service – “Attacker overwhelms a target with high‑volume tra
MIRAI-GREETH_FLOOD,MIRAI,DOS_FLOOD,FLOOD,FLOOD,false,true,"**Triage Summary** - **Predicted Attack Family:** **DOS_FLOOD** – high‑rate, uniform‑size traffic with no SYN flags, consistent with an ACK‑based denial‑of‑service flood. - **MI"
RECON-PORTSCAN,RECON,BENIGN,RECON_SCAN,BENIGN,false,false,"**Triage Summary** - **Predicted Attack Family:** **BENIGN** - **MITRE ATT&CK Context:** *Benign Traffic* – Normal, expected behavior with no malicious indicators. No specific tec"
VULNERABILITYSCAN,VULNERABILITYSCAN,VULNERABILITYSCAN,RECON_SCAN,RECON_SCAN,true,true,**Triage Summary** - **Predicted Attack Family:** VULNERABILITYSCAN - **MITRE ATT&CK Technique:** T1595.002 – Vulnerability Scanning (attacker probes systems for known weaknesse
MITM-ARPSPOOFING,SPOOFING,DDOS_FLOOD,SPOOFING,FLOOD,false,false,**Triage Summary** - **Predicted Attack Family:** DDOS_FLOOD - **MITRE ATT&CK Technique:** T1498 – Network Denial of Service (DDoS) - **Severity Score:** 10 / 10 (critical) –
DNS_SPOOFING,SPOOFING,WEB_ATTACK,SPOOFING,INTRUSION,false,false,"**Triage Summary** - **Predicted Attack Family:** WEB_ATTACK - **MITRE ATT&CK Technique:** T1190 – Exploitation of Public‑Facing Application (e.g., malicious HTTP requests, inje"
DICTIONARYBRUTEFORCE,BRUTEFORCE,RECON,INTRUSION,RECON_SCAN,false,false,"**Triage Summary** - **Predicted Attack Family:** RECON - **MITRE ATT&CK Technique:** T1595 – Active Scanning (reconnaissance of hosts, ports, and services) - **Severity Score"
SQLINJECTION,WEB_ATTACK,BENIGN,INTRUSION,BENIGN,false,false,"**Triage Summary** - **Predicted Attack Family:** **BENIGN** - **MITRE ATT&CK Context:** Benign Traffic – normal, expected device behavior with no malicious indicators. - **Se"


Exact family accuracy: 3/12 = 25%
Coarse triage accuracy: 5/12 = 42%


[Trace(trace_id=tr-43ec7e42de20b8c1f39869918f354305), Trace(trace_id=tr-10f8415a14c2f97ca76fecdf412d553f), Trace(trace_id=tr-5f9feabb42e95abdc2c0ac47128490dd), Trace(trace_id=tr-fc12e95a5c5d82e54e3e5575534bbb2c), Trace(trace_id=tr-e9eb3aeb14da882a2a32eb113bfefc26), Trace(trace_id=tr-01b0c5c3667d5d6c10269423bfdfe39f), Trace(trace_id=tr-c8de81f586218b452b1d2a024374a91d), Trace(trace_id=tr-c7f58b46ea2c7dcbd8a2d0ece5d62978), Trace(trace_id=tr-4c7e29628fd49d39876f951814c1951c), Trace(trace_id=tr-8c44a367071711d87b449060685afd9a)]